<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-02/AI_Modul_2.5_Praktikum_Operator_Matematika_dan_Logika.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 2.5: Praktikum Operator Matematika dan Logika
### Eksplorasi Aritmetika Presisi, Presedensi Operator, Evaluasi Hubung Singkat (Short-Circuit), dan Transformasi Fitur AI Perkebunan

---

> **Diktat Terkait:** [AI_Modul_2.5_Operator_Matematika_dan_Logika.md](../../docs/part-02/AI_Modul_2.5_Operator_Matematika_dan_Logika.md)  
> **Outputs:** Script uji pembagian Euclidean (`//` dan `%`), modul evaluasi presedensi operator, benchmark *short-circuit evaluation*, serta pipeline normalisasi fitur sensor (*Min-Max Scaling* dan *Z-Score Standardization*).  
> **Outcomes:** Mahasiswa terampil memilih dan menerapkan operator aritmetika, relasional, logika, identitas, dan keanggotaan secara presisi, mencegah bug presedensi kalkulasi dosis agrokimia, serta membangun transformasi fitur AI yang aman dari pembagian nol.  
> **Impacts:** Terciptanya sistem analitika presisi tinggi pada otomasi perkebunan modern, proteksi tanaman dari kegagalan formulasi agrokimia, dan efisiensi komputasi pada pemrosesan telemetri skala besar.
>
> **Tujuan Praktikum:**  
> 1. Memvalidasi mekanisme pembagian Euclidean, kuosien bulat (`//`), modulo (`%`), dan perilaku bilangan negatif di Python.  
> 2. Mengukur pengaruh presedensi operator dan pembuktian matematis pentingnya tanda kurung pada rumus agronomis.  
> 3. Mengamati perilaku *short-circuit evaluation* pada operator logika `and` dan `or`.  
> 4. Membandingkan perbedaan semantik dan performa antara kesetaraan nilai (`==`), identitas objek (`is`), dan operator keanggotaan (`in`).  
> 5. Membangun dan menguji sistem fertigasi analitika serta detektor anomali Z-score dinamis untuk telemetri PKS.

## 1. Persiapan Environment & Pemuatan Library
Jalankan sel berikut untuk memuat pustaka matematika, inspeksi sistem, dan konfigurasi grafik visualisasi:

In [ ]:
# Mengimpor modul matematika standar dan inspeksi sistem
import math
import sys
import time

# Mengimpor modul anotasi tipe data
from typing import List, Tuple, Dict, Any

# Mengimpor pustaka numerik dan visualisasi grafik
import numpy as np
import matplotlib
matplotlib.use('Agg') # Mode rendering non-interaktif yang stabil
import matplotlib.pyplot as plt

# Konfigurasi gaya visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("[OK] Seluruh pustaka berhasil dimuat dengan sukses.")
print(f"[INFO] Versi Python Aktif: {sys.version.split()[0]}")

## 2. Eksperimen Operator Aritmetika & Teorema Pembagian Euclidean
Operasi pembagian di Python dibedakan secara tegas menjadi pembagian riil (`/`), pembagian bulat ke bawah (*floor division*, `//`), dan modulo (`%`).
Rumus Euclidean: $a = b \times (a // b) + (a \% b)$.

In [ ]:
# Demonstrasi Aritmetika Dasar dan Pembagian Euclidean
pembilang_positif = 235.75
kapasitas_karung = 50.0

hasil_bagi_riil = pembilang_positif / kapasitas_karung
kuosien_bulat = pembilang_positif // kapasitas_karung
sisa_modulo = pembilang_positif % kapasitas_karung

print("=== PEMBAGIAN POSITIF (LOGISTIK PUPUK) ===")
print(f"Total Pupuk       : {pembilang_positif} kg")
print(f"Kapasitas Karung  : {kapasitas_karung} kg")
print(f"Pembagian Riil (/) : {hasil_bagi_riil:.4f}")
print(f"Kuosien Bulat (//): {kuosien_bulat} karung penuh")
print(f"Sisa Modulo (%)   : {sisa_modulo:.2f} kg sisa pupuk")

# Verifikasi Teorema Euclidean: a == b * (a // b) + (a % b)
rekonstruksi = (kapasitas_karung * kuosien_bulat) + sisa_modulo
print(f"Verifikasi Rekonstruksi : {rekonstruksi:.2f} == {pembilang_positif} -> {math.isclose(rekonstruksi, pembilang_positif)}")

# Eksperimen Modulo Bilangan Negatif (Circular Buffer Index)
print("\n=== PERILAKU MODULO BILANGAN NEGATIF ===")
langkah_anomali = -3
ukuran_buffer = 8
idx_python = langkah_anomali % ukuran_buffer
kuosien_negatif = langkah_anomali // ukuran_buffer

print(f"Langkah: {langkah_anomali}, Buffer Size: {ukuran_buffer}")
print(f"Python Floor Div (-3 // 8) : {kuosien_negatif}")
print(f"Python Modulo    (-3 % 8)  : {idx_python} (Aman untuk indeks array!)")
print("Catatan: Bahasa C/Java menghasilkan -3, yang dapat memicu IndexError pada array memori.")

## 3. Audit Presedensi Operator & Pengaruh Tanda Kurung
Pada sel ini kita akan mensimulasikan dampak fatal dari pengabaian presedensi operator pada formulasi dosis pestisida/pupuk cair.

In [ ]:
# Simulasi Formulasi Dosis Kimia Tangki Sprayer
pupuk_a = 50.0          # gram
pupuk_b = 50.0          # gram
faktor_kelarutan = 0.8  # efisiensi disolusi
volume_air = 200.0      # liter pelarut

# Versi A: Tanpa tanda kurung (Kesalahan Presedensi: * dan / dieksekusi sebelum +)
dosis_a = pupuk_a + pupuk_b * faktor_kelarutan / volume_air

# Versi B: Dengan tanda kurung eksplisit ((pupuk_a + pupuk_b) dikelompokkan dahulu)
dosis_b = (pupuk_a + pupuk_b) * faktor_kelarutan / volume_air

print("=== AUDIT PRESEDENSI FORMULASI DOSIS AGROKIMIA ===")
print(f"Konsentrasi Versi A (Salah - Tanpa Kurung) : {dosis_a:.4f} gram/liter")
print(f"Konsentrasi Versi B (Benar - Dengan Kurung): {dosis_b:.4f} gram/liter")
print(f"Faktor Kesalahan Relatif                    : {dosis_a / dosis_b:.2f}x lipat overdosis!")

# Evaluasi visual konsekuensi overdosis
ambang_toksisitas = 1.0  # gram/liter batas aman daun kelapa sawit
print(f"Status Versi A : {'[BAHAYA: DAUN TERBAKAR]' if dosis_a > ambang_toksisitas else '[AMAN]'}")
print(f"Status Versi B : {'[BAHAYA: DAUN TERBAKAR]' if dosis_b > ambang_toksisitas else '[AMAN - STANDAR AGRONOMI]'}")

## 4. Operator Relasional, Chained Comparison, & Short-Circuit Evaluation
Python memungkinkan penulisan relasi berantai matematis seperti `18 <= x <= 32`. Selain itu, operator logika `and` dan `or` bekerja menggunakan prinsip *Short-Circuit Evaluation* yang menghemat siklus komputasi.

In [ ]:
# Uji Perbandingan Berantai (Chained Comparison)
suhu_greenhouse = 26.5  # Celcius
kelembaban_rh = 78.0    # Persentase RH

# Sintaks Tradisional vs Sintaks Berantai Python
valid_tradisional = (suhu_greenhouse >= 20.0) and (suhu_greenhouse <= 30.0)
valid_berantai = 20.0 <= suhu_greenhouse <= 30.0

print("=== UJI PERBANDINGAN BERANTAI ===")
print(f"Suhu Aktual      : {suhu_greenhouse} C")
print(f"Valid Tradisional: {valid_tradisional}")
print(f"Valid Berantai   : {valid_berantai}")

# Uji Evaluasi Hubung Singkat (Short-Circuit Evaluation)
print("\n=== UJI SHORT-CIRCUIT EVALUATION ===")

def cek_koneksi_gateway() -> bool:
    print("  -> [FUNGSI DIPANGGIL] Memeriksa sinyal RF gateway...")
    return False  # Gateway offline

def baca_sensor_berat() -> float:
    print("  -> [FUNGSI DIPANGGIL] Membaca sensor telemetri (komputasi berat)...\a")
    return 45.2

print("Skenario 1: Gerbang Logika 'AND' dengan kondisi kiri False:")
# Jika koneksi False, fungsi baca_sensor_berat TIDAK AKAN PERNAH dipanggil!
status_baca = cek_koneksi_gateway() and (baca_sensor_berat() > 40.0)
print(f"Hasil Evaluasi Akhir: {status_baca} (Sensor berat diabaikan demi efisiensi)")

print("\nSkenario 2: Gerbang Logika 'OR' dengan kondisi kiri True:")
def status_daya_darurat() -> bool:
    print("  -> [FUNGSI DIPANGGIL] Cek baterai cadangan...")
    return True

status_daya = status_daya_darurat() or cek_koneksi_gateway()
print(f"Hasil Evaluasi Akhir: {status_daya} (Gateway dilewati karena daya sudah True)")

## 5. Kesetaraan Nilai (`==`), Identitas Objek (`is`), dan Keanggotaan (`in`)
Memahami perbedaan operator identitas memori dan pencarian keanggotaan koleksi data berkecepatan tinggi.

In [ ]:
# Uji Kesetaraan Nilai (==) vs Identitas Memori (is)
list_sensor_a = [10.5, 20.3, 30.8]
list_sensor_b = [10.5, 20.3, 30.8]
list_sensor_c = list_sensor_a

print("=== KESETARAAN (==) VS IDENTITAS MEMORI (is) ===")
print(f"list_sensor_a == list_sensor_b : {list_sensor_a == list_sensor_b} (Isi nilai sama)")
print(f"list_sensor_a is list_sensor_b : {list_sensor_a is list_sensor_b} (Alamat memori berbeda!)")
print(f"list_sensor_a is list_sensor_c : {list_sensor_a is list_sensor_c} (Menunjuk ke objek memori yang sama)")
print(f"id(a): {id(list_sensor_a)} | id(b): {id(list_sensor_b)} | id(c): {id(list_sensor_c)}")

# Benchmark Kecepatan Operator Keanggotaan 'in': List vs Set
print("\n=== BENCHMARK OPERATOR KEANGGOTAAN 'in' (LIST VS SET) ===")
jumlah_elemen = 200000
data_list = list(range(jumlah_elemen))
data_set = set(data_list)
target_cari = 199999  # Elemen paling akhir (worst-case untuk list O(N))

# Waktu pencarian pada List
t0 = time.perf_counter()
ketemu_list = target_cari in data_list
t_list = (time.perf_counter() - t0) * 1000.0  # milidetik

# Waktu pencarian pada Set (Hash Table)
t0 = time.perf_counter()
ketemu_set = target_cari in data_set
t_set = (time.perf_counter() - t0) * 1000.0  # milidetik

print(f"Pencarian elemen pada List (O(N))    : {t_list:.4f} ms")
print(f"Pencarian elemen pada Set  (O(1))    : {t_set:.6f} ms")
print(f"Rasio Percepatan Menggunakan Set     : {t_list / max(t_set, 1e-9):.1f}x lebih cepat!")

## 6. Kasus Terintegrasi: Mesin Analitika Fertirigasi & Normalisasi Fitur AI
Implementasi kelas industri `FertigationAnalyticsEngine` dari Diktat Bagian 5 yang menggabungkan operator penugasan majemuk, pembagian Euclidean, operator keanggotaan cepat, serta transformasi Min-Max dan Z-Score.

In [ ]:
# Implementasi Mesin Analitika Fertirigasi Presisi
class FertigationAnalyticsEngine:
    """
    Engine komputasi cerdas untuk mengontrol debit air, partisi pupuk,
    dan normalisasi telemetri sensor lingkungan perkebunan.
    """
    def __init__(self, kapasitas_tangki_liter: float) -> None:
        self.kapasitas_tangki = float(kapasitas_tangki_liter)
        self.volume_air_saat_ini = 0.0
        self.pupuk_terdaftar = {"UREA-46", "SP36-SUPER", "MOP-60", "ZA-NON-SUBSIDI", "NPK-15-15-15"}

    def isi_air_tangki(self, debit_liter_per_detik: float, durasi_detik: float) -> float:
        volume_masuk = debit_liter_per_detik * durasi_detik
        self.volume_air_saat_ini += volume_masuk  # Operator penugasan majemuk
        if self.volume_air_saat_ini > self.kapasitas_tangki:
            self.volume_air_saat_ini = self.kapasitas_tangki
        return self.volume_air_saat_ini

    def kalkulasi_kemasan_dan_sisa(self, kebutuhan_kg: float, kapasitas_karung_kg: float = 50.0) -> Tuple[int, float]:
        jumlah_karung = int(kebutuhan_kg // kapasitas_karung_kg)  # Floor division
        sisa_lepas = round(kebutuhan_kg % kapasitas_karung_kg, 4)  # Modulo
        return jumlah_karung, sisa_lepas

    def verifikasi_legalitas_pupuk(self, kode_formula: str) -> bool:
        return kode_formula.strip().upper() in self.pupuk_terdaftar  # Operator keanggotaan O(1)

    def normalisasi_min_max(self, data: List[float]) -> List[float]:
        if not data:
            return []
        val_min = min(data)
        val_max = max(data)
        rentang = val_max - val_min
        if math.isclose(rentang, 0.0):
            return [0.0] * len(data)  # Zero division guard
        return [round((x - val_min) / rentang, 4) for x in data]

    def standarisasi_z_score(self, data: List[float]) -> List[float]:
        if not data:
            return []
        n = len(data)
        rata_rata = sum(data) / n
        varians = sum((x - rata_rata) ** 2 for x in data) / n
        std_dev = math.sqrt(varians)
        if math.isclose(std_dev, 0.0):
            return [0.0] * n  # Zero division guard
        return [round((x - rata_rata) / std_dev, 4) for x in data]

# Inisialisasi dan Pengujian Engine
engine = FertigationAnalyticsEngine(kapasitas_tangki_liter=5000.0)
air_terisi = engine.isi_air_tangki(debit_liter_per_detik=12.5, durasi_detik=180.0)
karung_urea, sisa_urea = engine.kalkulasi_kemasan_dan_sisa(235.75, kapasitas_karung_kg=50.0)

print("=== EKSEKUSI ENGINE FERTIRIGASI ANALITIKA ===")
print(f"1. Volume Air Terisi        : {air_terisi:.1f} / 5000.0 Liter")
print(f"2. Partisi Karung Urea      : {karung_urea} Karung Penuh + Sisa {sisa_urea} kg")

# Uji Keanggotaan Formula Pupuk
daftar_uji = ["UREA-46", "ZA-NON-SUBSIDI", "PESTISIDA-ILEGAL"]
print("\n3. Verifikasi Keabsahan Pupuk:")
for pupuk in daftar_uji:
    legal = engine.verifikasi_legalitas_pupuk(pupuk)
    print(f"   - {pupuk:<18} : {'[RESMI]' if legal else '[DITOLAK]'}")

# Uji Normalisasi Fitur Telemetri Sensor RH
telemetri_rh = [32.0, 44.5, 55.0, 68.2, 75.0, 82.5, 90.0]
rh_minmax = engine.normalisasi_min_max(telemetri_rh)
rh_zscore = engine.standarisasi_z_score(telemetri_rh)

print("\n4. Transformasi Fitur Telemetri RH (%):")
print(f"   Data Mentah : {telemetri_rh}")
print(f"   Min-Max [0,1]: {rh_minmax}")
print(f"   Z-Score (std): {rh_zscore}")

## 7. Solusi Tantangan Mandiri (Scaffolded Challenges)
Di bawah ini disajikan implementasi solusi lengkap untuk ketiga tantangan praktikum berjenjang.

In [ ]:
# === TANTANGAN 1: KONVERSI SATUAN SUHU DAN TITIK EMBUN ===
def fahrenheit_ke_celsius(f_val: float) -> float:
    """
    Mengonversi nilai temperatur Fahrenheit ke Celsius secara presisi.
    Presedensi operator dilindungi oleh tanda kurung: (F - 32) * (5/9).
    """
    c_val = (f_val - 32.0) * (5.0 / 9.0)
    return round(c_val, 2)

# Pengujian titik beku dan titik didih air
print("=== TANTANGAN 1: KONVERSI FAHRENHEIT KE CELSIUS ===")
print(f"Titik Beku (32.0 F)  -> {fahrenheit_ke_celsius(32.0)} C  (Ekspektasi: 0.00 C)")
print(f"Titik Didih (212.0 F) -> {fahrenheit_ke_celsius(212.0)} C (Ekspektasi: 100.00 C)")
print(f"Suhu Lapangan (86.0 F)-> {fahrenheit_ke_celsius(86.0)} C  (Ekspektasi: 30.00 C)")

In [ ]:
# === TANTANGAN 2: MINI-BATCH SLICER UNTUK PELATIHAN MODEL AI ===
def hitung_partisi_batch(total_sampel: int, batch_size: int) -> Dict[str, int]:
    """
    Menghitung partisi mini-batch pelatihan citra daun sawit menggunakan
    operator pembagian Euclidean murni tanpa percabangan if.
    """
    batch_penuh = total_sampel // batch_size
    sisa_sampel = total_sampel % batch_size
    # Jika sisa_sampel > 0, bernilai 1; jika 0, bernilai 0 (Trik Boolean-to-Integer)
    total_iterasi = batch_penuh + int(sisa_sampel > 0)
    
    return {
        "total_sampel": total_sampel,
        "batch_size": batch_size,
        "batch_penuh": batch_penuh,
        "sisa_sampel_terakhir": sisa_sampel,
        "total_iterasi_per_epoch": total_iterasi
    }

print("\n=== TANTANGAN 2: PARTISI MINI-BATCH CITRA SAWIT ===")
hasil_partisi = hitung_partisi_batch(total_sampel=1054, batch_size=32)
for k, v in hasil_partisi.items():
    print(f"{k:<25}: {v}")

In [ ]:
# === TANTANGAN 3: DETEKTOR ANOMALI SENSOR DENGAN Z-SCORE DINAMIS ===
class DetectorAnomaliZScore:
    """
    Detektor anomali gas metana (CH4) berbasis aturan 3-sigma Z-Score
    menggunakan operator relasional dan logika.
    """
    def __init__(self, ambang_batas_z: float = 3.0) -> None:
        self.ambang_batas = float(ambang_batas_z)
        self.riwayat_sensor: List[float] = []

    def perbarui_dan_deteksi(self, nilai_baru: float) -> Tuple[float, bool]:
        self.riwayat_sensor.append(nilai_baru)
        n = len(self.riwayat_sensor)
        
        if n < 3: # Butuh minimal 3 data untuk varians bermakna
            return 0.0, False
            
        mu = sum(self.riwayat_sensor) / n
        var = sum((x - mu) ** 2 for x in self.riwayat_sensor) / n
        sigma = math.sqrt(var)
        
        if math.isclose(sigma, 0.0):
            return 0.0, False
            
        z_score = (nilai_baru - mu) / sigma
        # Kondisi anomali menggunakan operator logika 'or' atau 'abs()'
        is_anomali = (z_score > self.ambang_batas) or (z_score < -self.ambang_batas)
        return round(z_score, 2), is_anomali

# Simulasi Aliran Data Sensor Gas Metana PKS
np.random.seed(42)
data_normal = np.random.normal(loc=15.0, scale=1.2, size=50).tolist()
# Sisipkan anomali kebocoran pipa gas pada langkah ke-35 dan ke-45
data_normal[35] = 24.8  # Lonjakan anomali gas
data_normal[45] = 27.5  # Lonjakan kritis

detektor = DetectorAnomaliZScore(ambang_batas_z=3.0)
z_scores = []
daftar_anomali = []

for t, reading in enumerate(data_normal):
    z, is_anom = detektor.perbarui_dan_deteksi(reading)
    z_scores.append(z)
    if is_anom:
        daftar_anomali.append((t, reading, z))

print("\n=== TANTANGAN 3: DETEKSI ANOMALI GAS METANA PKS ===")
print(f"Total Pembacaan Sensor: {len(data_normal)}")
print(f"Jumlah Titik Anomali  : {len(daftar_anomali)}")
for t, val, z in daftar_anomali:
    print(f"  [ALERT] Indeks {t:02d} | Konsentrasi: {val:.2f} ppm | Z-Score: {z:+.2f}")

# Visualisasi Grafik Telemetri & Z-Score
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Plot 1: Nilai Mentah Sensor Gas Metana
ax1.plot(data_normal, color='teal', lw=1.5, label='Konsentrasi CH4 (ppm)')
for t, val, _ in daftar_anomali:
    ax1.scatter(t, val, color='red', s=70, zorder=5, label='Anomali Terdeteksi' if t == daftar_anomali[0][0] else "")
ax1.set_ylabel('Konsentrasi (ppm)')
ax1.set_title('Telemetri Sensor Gas Metana (CH4) Kolam Limbah PKS', fontweight='bold')
ax1.legend(loc='upper left')

# Plot 2: Kurva Skor Z-Score Dinamis & Ambang Batas
ax2.plot(z_scores, color='purple', lw=1.5, label='Z-Score Dinamis')
ax2.axhline(3.0, color='red', linestyle='--', lw=1.2, label='Batas Atas (+3 Sigma)')
ax2.axhline(-3.0, color='red', linestyle='--', lw=1.2, label='Batas Bawah (-3 Sigma)')
ax2.set_xlabel('Sampel Telemetri Waktu-Nyata (Detik)')
ax2.set_ylabel('Z-Score')
ax2.set_title('Evaluasi Ambang Batas Logika Komparasi (|Z| > 3.0)', fontweight='bold')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.savefig('docs/assets/simulasi_deteksi_anomali_zscore.png', dpi=150)
print("[INFO] Grafik deteksi anomali berhasil dirender ke 'docs/assets/simulasi_deteksi_anomali_zscore.png'.")